## homework 3 - Functions I

1. Write a function that takes a single numerical variable and returns three values, the minimum number, the median, and the maximum number of the vector. Test your function using the month column of the flights data set.

In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("nycflights.csv")

In [3]:
df.head()

,Unnamed: 0,year,month,day,dep_time,sched_dep_time,dep_delay,arr_time,sched_arr_time,arr_delay,carrier,flight,tailnum,origin,dest,air_time,distance,hour,minute,time_hour
0,1,2013,1,1,517.0,515,2.0,830.0,819,11.0,UA,1545,N14228,EWR,IAH,227.0,1400,5,15,2013-01-01 05:00:00
1,2,2013,1,1,533.0,529,4.0,850.0,830,20.0,UA,1714,N24211,LGA,IAH,227.0,1416,5,29,2013-01-01 05:00:00
2,3,2013,1,1,542.0,540,2.0,923.0,850,33.0,AA,1141,N619AA,JFK,MIA,160.0,1089,5,40,2013-01-01 05:00:00
3,4,2013,1,1,544.0,545,-1.0,1004.0,1022,-18.0,B6,725,N804JB,JFK,BQN,183.0,1576,5,45,2013-01-01 05:00:00
4,5,2013,1,1,554.0,600,-6.0,812.0,837,-25.0,DL,461,N668DN,LGA,ATL,116.0,762,6,0,2013-01-01 06:00:00


In [7]:
def summary(x):
        return {
        "min": np.min(x),
        "median": np.median(x),
        "max": np.max(x)
    }

In [9]:
summary(df['month'])

{'min': 1, 'median': np.float64(7.0), 'max': 12}

2. Explain your reasoning for choosing your function's name in the previous question.

Because it returns the min, median, and max, so it is called 'summary'.

3. Write a function that categorizes a numerical variable in the flights data into four categories.

   The function should have two arguments. The first should represent the data object and the second should represent a column name in the data object.
   
   The function should return one new column which categorizes the dep_time column into four categories in the following manner. For any particular variable in the flights data that represents military time (i.e., 0 to 2400 where 1200 represents 12 in the afternoon and 2400 represents midnight), the function should classify values into four categories:

   "Morning" for values from 5 am to 11:59 am

   "Afternoon" for values from 12 pm to 4:59 pm

   "Evening" for values from 5 pm to 8:59 pm

   "Night" for values from 9 pm to 4:59 am

   Test your function using the dep_time column of the flights dataset. Print a frequency table of the output.

In [16]:
def categorize(data, col):
    conditions = [(data[col] >= 500) & (data[col] <= 1159), (data[col] >= 1200) & (data[col] <= 1659), (data[col] >= 1700) & (data[col] <= 2059), (data[col] >= 2100) | (data[col] <= 459)]
    labels = ["Morning", "Afternoon", "Evening", "Night"]
    new_column = pd.Series(
        np.select(conditions, labels, default=None),
    )
    return new_column

In [17]:
dep_time_cat = categorize(df, "dep_time")

In [18]:
print(dep_time_cat.value_counts(dropna=False))

Morning      129539
Afternoon     98617
Evening       79793
Night         20572
None           8255
Name: count, dtype: int64


4. Explain your reasoning for choosing your function's name in the previous question.

Because the function categorize the data into four categories, so it is called 'categorize'.

5. Write a function that calculates the median of all numeric variables in the flights dataset.

Hint: There are several ways to subset a DF to numeric values only, you will need to search online (using google, stackexchange or some generative AI like chat GPT) to find a suitable way to do so. Alternatively, you could try to do this manually with a for loop, which was reviewed in datacamp, although as a class we are not learning for loops together until next week.

In [20]:
def median_all(data):
    numeric = data.select_dtypes(include="number")
    return numeric.median()

In [21]:
median_all(df)

Unnamed: 0        168388.5
year                2013.0
month                  7.0
day                   16.0
dep_time            1401.0
sched_dep_time      1359.0
dep_delay             -2.0
arr_time            1535.0
sched_arr_time      1556.0
arr_delay             -5.0
flight              1496.0
air_time             129.0
distance             872.0
hour                  13.0
minute                29.0
dtype: float64

6. Explain your reasoning for choosing your function's name in the previous question.

Because this function calculate all the median number, so it is called 'median_all'.

7.  Modify the function t_test() we wrote in class together this week so that this function can handle violations to the homogeneity of variance (HOV) assumption.

In [22]:
from scipy import stats

def t_test(num_var, bin_var):
    
    group1 = num_var[bin_var == 0]
    group2 = num_var[bin_var == 1]
    
    n1, n2 = group1.shape[0], group2.shape[0]
    mean1, mean2 = group1.mean(), group2.mean()
    var1, var2 = group1.var(ddof=1), group2.var(ddof=1)
    
    mean_diff = mean2 - mean1
    
    var_ratio = var1 / var2 if var2 != 0 else np.inf
    
    hov_ok = 0.25 <= var_ratio <= 4

    if hov_ok:
        DF = n1 + n2 - 2
        
        sp = np.sqrt(((n1 - 1)*var1 + (n2 - 1)*var2) / DF)
        SE_mean_diff = sp * np.sqrt(1/n1 + 1/n2)
        t_statistic = mean_diff / SE_mean_diff
        
        test_name = "Independent samples t-test"

    else:
        
        SE_mean_diff = np.sqrt(var1/n1 + var2/n2)
        t_statistic = mean_diff / SE_mean_diff
        
        # Welch–Satterthwaite df
        DF = (var1/n1 + var2/n2)**2 / (
            ((var1/n1)**2) / (n1 - 1) +
            ((var2/n2)**2) / (n2 - 1)
        )
        
        test_name = "Welch's t-test"
    
    p_value = 2 * (1 - stats.t.cdf(np.abs(t_statistic), DF))
    
    results = pd.DataFrame({
        "Continuous Variable": [num_var.name],
        "Binary Variable": [bin_var.name],
        "Total Sample Size": [n1 + n2],
        "Mean Difference": [round(mean_diff, 2)],
        "SE of Mean Difference": [round(SE_mean_diff, 2)],
        "DF": [round(DF, 2)],
        "t-statistic": [round(t_statistic, 3)],
        "P-value": [("%.3f" % p_value).lstrip('0')],
        "Test": [test_name]
    })
    
    return results

8. Import the GED data set we used for the class activity. Call the t_test() function and test it out on the GED data set we used in class. Let the numeric variable be 'income_log' and let the binary variable be 'ged'.

In [23]:
ged_df = pd.read_csv("3 ged_data.csv")

In [24]:
t_test(ged_df["income_log"], ged_df["ged"])

,Continuous Variable,Binary Variable,Total Sample Size,Mean Difference,SE of Mean Difference,DF,t-statistic,P-value,Test
0,income_log,ged,5976,-0.48,0.06,5974,-7.458,.000,Independent samples t-test
